# Membuat dan memeriksa submission

Saya memuat stack hasil notebook 02 dan membuat kandidat baru dari data test.
File `submission.csv` dan `best_stack_087.csv` yang sudah ada tetap dipertahankan.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts').exists(), 'Jalankan dari folder proyek atau notebooks.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / 'data'
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
SEED = 2026
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
plt.rcParams['figure.dpi'] = 110

## 1. Memuat model dan menghasilkan prediksi

Fungsi prediksi memakai preprocessing yang tersimpan, tanpa mempelajari statistik dari test.
Prediksi dilakukan per bagian untuk membatasi penggunaan memori. Urutan template dipertahankan.

In [ ]:
import hashlib
from scripts.predict import create_submission, validate_submission

protected = [ROOT / 'submissions' / name for name in ['submission.csv', 'best_stack_087.csv']]
before = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
model_path = ROOT / 'models' / 'reproducible_stack.joblib'
assert model_path.exists(), 'Jalankan notebook 02 terlebih dahulu.'
output_path = ROOT / 'submissions' / 'candidate_reproducible.csv'
submission = create_submission(model_path, DATA_DIR, output_path)
print('Kandidat baru:', output_path.relative_to(ROOT))
display(submission.head())

## 2. Memeriksa format, ID, dan label

File harus memiliki tepat dua kolom, satu prediksi per ID, label integer 1–5, serta urutan yang sama
dengan sample submission dan test. Pemeriksaan dilakukan lagi setelah membaca CSV yang disimpan.

In [ ]:
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
saved = pd.read_csv(output_path)
validate_submission(saved, sample, test)
assert len(saved) == 228629
assert saved.equals(submission)
after = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}
assert before == after, 'CSV final lama berubah.'
print('Semua pemeriksaan berhasil.')
print('Jumlah baris:', len(saved))
print('Kolom:', saved.columns.tolist())
print('ID unik:', saved.building_id.is_unique)
print('CSV final lama tetap identik:', before == after)

## 3. Melihat distribusi prediksi

Saya membandingkan proporsi prediksi dengan target training untuk mendeteksi perubahan yang mencolok.
Distribusi serupa bukan bukti akurasi, dan distribusi berbeda tidak otomatis berarti prediksi salah.
Label test tidak tersedia sehingga Macro F1 test tidak dihitung di sini.

In [ ]:
train_labels = pd.read_csv(DATA_DIR / 'train.csv', usecols=['damage_grade'])['damage_grade']
train_codes = train_labels.astype(str).str.extract(r'([1-5])$', expand=False).astype(int)
comparison = pd.DataFrame({
    'training': train_codes.value_counts(normalize=True).sort_index(),
    'prediksi_kandidat': saved.damage_grade.value_counts(normalize=True).reindex(range(1, 6), fill_value=0),
})
display(comparison.round(4))
ax = comparison.plot.bar(figsize=(8, 4), rot=0)
ax.set(title='Proporsi kelas training dan prediksi kandidat', xlabel='Damage grade', ylabel='Proporsi')
plt.tight_layout()
plt.savefig(FIGURES / 'submission_distribution.png', bbox_inches='tight')
plt.show()

## 4. Pengumpulan

Jalankan sel di atas untuk menyimpan kandidat sebagai `submissions/candidate_reproducible.csv`.
Jika dipilih untuk dikumpulkan, unggah file tersebut secara manual pada halaman kompetisi setelah
menerima aturan. Notebook ini tidak melakukan upload atau mengganti CSV final lama.
Laporan evaluasi tersedia pada notebook 02 dan folder reports.